In [1]:
!pip install -q torchmetrics opencv-python scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 18.1 MB/s eta 0:00:00


In [2]:
import os
import cv2
import json
import yaml
import math
import random
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import nms
from torchmetrics.detection.mean_ap import MeanAveragePrecision

from sklearn.model_selection import KFold
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


## 1. 원본 데이터 압축 해제

In [4]:
dataset_path = '/content/drive/MyDrive/ai09-level1-project.zip'
output_dir = '/content/ai09-project01/'
os.makedirs(output_dir, exist_ok=True)

with zipfile.ZipFile(dataset_path) as zip_file:
    zip_file.extractall(path=output_dir)

data_root = "/content/ai09-project01/sprint_ai_project1_data"
train_image_dir = os.path.join(data_root, "train_images")
train_annotation_dir = os.path.join(data_root, "train_annotations")
test_image_dir = os.path.join(data_root, "test_images")

print("Train image dir:", train_image_dir)
print("Train annotation dir:", train_annotation_dir)
print("Test image dir:", test_image_dir)

Train image dir: /content/ai09-project01/sprint_ai_project1_data/train_images
Train annotation dir: /content/ai09-project01/sprint_ai_project1_data/train_annotations
Test image dir: /content/ai09-project01/sprint_ai_project1_data/test_images


## 2. annotation 파싱 / YOLO 변환

In [5]:
def safe_get(d, keys, default=None):
    for k in keys:
        if k in d and d[k] not in [None, ""]:
            return d[k]
    return default

def get_image_files(image_dir, exts=(".png", ".jpg", ".jpeg")):
    image_files = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith(exts)
    ])
    print("Number of image files:", len(image_files))
    print("Sample image files:", image_files[:5])
    return image_files

def build_images_df(image_dir, image_files):
    rows = []

    for file_name in image_files:
        img_path = os.path.join(image_dir, file_name)
        img = cv2.imread(img_path)

        if img is None:
            rows.append({
                "file_name": file_name,
                "width": None,
                "height": None,
                "channel": None,
                "is_broken": True
            })
            continue

        h, w, c = img.shape
        rows.append({
            "file_name": file_name,
            "width": w,
            "height": h,
            "channel": c,
            "is_broken": False
        })

    images_df = pd.DataFrame(rows)
    images_df["image_key"] = images_df["file_name"].apply(lambda x: os.path.splitext(x)[0])
    return images_df

def parse_annotation_json(json_path, image_folder_name=None):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []

    if isinstance(data, dict) and "annotations" in data:
        annotations = data.get("annotations", [])
        images = data.get("images", [])
        categories = data.get("categories", [])

        image_info = images[0] if len(images) > 0 else {}
        image_id = safe_get(image_info, ["id"], default=None)
        file_name = safe_get(image_info, ["file_name"], default=None)

        category_map = {}
        for cat in categories:
            cat_id = safe_get(cat, ["id"], default=None)
            cat_name = safe_get(cat, ["name"], default=None)
            category_map[cat_id] = cat_name

        for ann in annotations:
            bbox = safe_get(ann, ["bbox"], default=None)
            category_id = safe_get(ann, ["category_id"], default=None)

            rows.append({
                "image_key": image_folder_name,
                "json_file": os.path.basename(json_path),
                "file_name_from_json": file_name,
                "image_id": image_id,
                "bbox": bbox,
                "category_id": category_id,
                "category_name": category_map.get(category_id, None),
                "dl_idx": safe_get(image_info, ["dl_idx"]),
                "dl_name": safe_get(image_info, ["dl_name"]),
                "drug_shape": safe_get(image_info, ["drug_shape"]),
                "color_class1": safe_get(image_info, ["color_class1"]),
                "color_class2": safe_get(image_info, ["color_class2"]),
                "line_front": safe_get(image_info, ["line_front"]),
                "line_back": safe_get(image_info, ["line_back"]),
                "print_front": safe_get(image_info, ["print_front"]),
                "print_back": safe_get(image_info, ["print_back"]),
                "json_path": json_path
            })
        return rows

    raise ValueError(f"Unsupported annotation format: {json_path}")

def build_annotations_df(annotation_root):
    rows = []

    for root, dirs, files in os.walk(annotation_root):
        json_files = [f for f in files if f.lower().endswith(".json")]
        for jf in json_files:
            json_path = os.path.join(root, jf)
            rel_dir = os.path.relpath(root, annotation_root)
            dir_parts = rel_dir.split(os.sep)

            json_stem = os.path.splitext(jf)[0]
            image_key = json_stem

            try:
                parsed_rows = parse_annotation_json(json_path, image_folder_name=image_key)
                for row in parsed_rows:
                    row["relative_dir"] = rel_dir
                    row["dir_depth"] = len(dir_parts)
                    rows.append(row)
            except Exception as e:
                rows.append({
                    "image_key": image_key,
                    "json_file": jf,
                    "file_name_from_json": None,
                    "image_id": None,
                    "bbox": None,
                    "category_id": None,
                    "category_name": None,
                    "dl_idx": None,
                    "dl_name": None,
                    "json_path": json_path,
                    "relative_dir": rel_dir,
                    "dir_depth": len(dir_parts),
                    "parse_error": str(e)
                })

    annotations_df = pd.DataFrame(rows)
    print("annotations_df shape:", annotations_df.shape)
    return annotations_df

In [6]:
def convert_loaded_annotations_to_yolo(
    images_df,
    annotations_df,
    train_image_dir,
    output_root,
    class_source="auto",
    copy_images=True,
    split_name="train"
):
    def is_valid_bbox(bbox):
        return isinstance(bbox, list) and len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0

    def xywh_to_yolo(img_w, img_h, bbox):
        x, y, w, h = bbox
        x_center = (x + w / 2.0) / img_w
        y_center = (y + h / 2.0) / img_h
        w = w / img_w
        h = h / img_h
        return x_center, y_center, w, h

    def choose_class_series(df, class_source):
        if class_source == "category_name" and "category_name" in df.columns:
            return df["category_name"]
        elif class_source == "dl_name" and "dl_name" in df.columns:
            return df["dl_name"]
        elif class_source == "category_id" and "category_id" in df.columns:
            return df["category_id"].astype(str)
        elif class_source == "dl_idx" and "dl_idx" in df.columns:
            return df["dl_idx"].astype(str)
        elif class_source == "auto":
            if "category_name" in df.columns and df["category_name"].notna().sum() > 0:
                return df["category_name"]
            elif "dl_name" in df.columns and df["dl_name"].notna().sum() > 0:
                return df["dl_name"]
            elif "category_id" in df.columns and df["category_id"].notna().sum() > 0:
                return df["category_id"].astype(str)
            elif "dl_idx" in df.columns and df["dl_idx"].notna().sum() > 0:
                return df["dl_idx"].astype(str)
            else:
                raise ValueError("No valid class column found in annotations_df.")
        else:
            raise ValueError(f"Invalid class_source: {class_source}")

    image_out_dir = os.path.join(output_root, "images", split_name)
    label_out_dir = os.path.join(output_root, "labels", split_name)
    os.makedirs(image_out_dir, exist_ok=True)
    os.makedirs(label_out_dir, exist_ok=True)

    merged_df = annotations_df.copy()
    class_series = choose_class_series(merged_df, class_source).fillna("UNKNOWN").astype(str)
    unique_classes = sorted(class_series.unique().tolist())
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(unique_classes)}

    image_meta = images_df.set_index("image_key")[["file_name", "width", "height"]].to_dict("index")
    grouped = merged_df.groupby("image_key")

    valid_image_count = 0
    valid_box_count = 0

    for image_key, group in grouped:
        if image_key not in image_meta:
            continue

        file_name = image_meta[image_key]["file_name"]
        img_w = image_meta[image_key]["width"]
        img_h = image_meta[image_key]["height"]

        if pd.isna(img_w) or pd.isna(img_h):
            continue

        label_lines = []
        for _, row in group.iterrows():
            bbox = row.get("bbox", None)
            if not is_valid_bbox(bbox):
                continue

            cls_name = str(choose_class_series(group, class_source).loc[row.name]) if row.name in group.index else None
            if cls_name is None:
                continue

            cls_id = class_to_id[cls_name]
            x_c, y_c, w, h = xywh_to_yolo(img_w, img_h, bbox)
            label_lines.append(f"{cls_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")
            valid_box_count += 1

        if len(label_lines) == 0:
            continue

        src_img_path = os.path.join(train_image_dir, file_name)
        dst_img_path = os.path.join(image_out_dir, file_name)
        dst_lbl_path = os.path.join(label_out_dir, f"{image_key}.txt")

        if copy_images and os.path.exists(src_img_path):
            shutil.copy2(src_img_path, dst_img_path)

        with open(dst_lbl_path, "w", encoding="utf-8") as f:
            f.write("\n".join(label_lines))

        valid_image_count += 1

    names_dict = {idx: name for name, idx in class_to_id.items()}
    yaml_path = os.path.join(output_root, "dataset.yaml")
    yaml_data = {
        "path": output_root,
        "train": f"images/{split_name}",
        "val": f"images/{split_name}",
        "names": names_dict
    }

    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(yaml_data, f, sort_keys=False, allow_unicode=True)

    print(f"YOLO dataset created at: {output_root}")
    print("Valid images:", valid_image_count)
    print("Valid boxes:", valid_box_count)
    print("Number of classes:", len(class_to_id))

    return {
        "image_dir": image_out_dir,
        "label_dir": label_out_dir,
        "yaml_path": yaml_path,
        "class_to_id": class_to_id
    }


def make_yolo_kfold_dataset(
    pairs,
    image_dir,
    label_dir,
    output_root,
    class_to_id,
    n_splits=5,
    random_state=42,
    shuffle=True
):
    os.makedirs(output_root, exist_ok=True)

    names_dict = {idx: name for name, idx in class_to_id.items()}
    indices = list(range(len(pairs)))
    kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)

    fold_yaml_paths = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(indices), start=1):
        fold_root = os.path.join(output_root, f"fold_{fold_idx}")
        fold_train_img_dir = os.path.join(fold_root, "images", "train")
        fold_val_img_dir = os.path.join(fold_root, "images", "val")
        fold_train_lbl_dir = os.path.join(fold_root, "labels", "train")
        fold_val_lbl_dir = os.path.join(fold_root, "labels", "val")

        os.makedirs(fold_train_img_dir, exist_ok=True)
        os.makedirs(fold_val_img_dir, exist_ok=True)
        os.makedirs(fold_train_lbl_dir, exist_ok=True)
        os.makedirs(fold_val_lbl_dir, exist_ok=True)

        for idx in train_idx:
            img_file, lbl_file = pairs[idx]
            shutil.copy2(os.path.join(image_dir, img_file), os.path.join(fold_train_img_dir, img_file))
            shutil.copy2(os.path.join(label_dir, lbl_file), os.path.join(fold_train_lbl_dir, lbl_file))

        for idx in val_idx:
            img_file, lbl_file = pairs[idx]
            shutil.copy2(os.path.join(image_dir, img_file), os.path.join(fold_val_img_dir, img_file))
            shutil.copy2(os.path.join(label_dir, lbl_file), os.path.join(fold_val_lbl_dir, lbl_file))

        yaml_path = os.path.join(fold_root, "dataset.yaml")
        yaml_data = {
            "path": fold_root,
            "train": "images/train",
            "val": "images/val",
            "names": names_dict
        }
        with open(yaml_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(yaml_data, f, sort_keys=False, allow_unicode=True)

        fold_yaml_paths.append(yaml_path)

    print("Created fold yaml files:")
    for p in fold_yaml_paths:
        print(" -", p)

    return fold_yaml_paths

## 3. YOLO 형식 데이터셋 생성 + K-Fold 분할

In [38]:
image_files = get_image_files(train_image_dir)
images_df = build_images_df(train_image_dir, image_files)
annotations_df = build_annotations_df(train_annotation_dir)

annotations_df = annotations_df.copy()
if "file_name_from_json" in annotations_df.columns:
    mask = annotations_df["file_name_from_json"].notna()
    annotations_df.loc[mask, "image_key"] = annotations_df.loc[mask, "file_name_from_json"].apply(
        lambda x: os.path.splitext(os.path.basename(x))[0]
    )

yolo_info = convert_loaded_annotations_to_yolo(
    images_df=images_df,
    annotations_df=annotations_df,
    train_image_dir=train_image_dir,
    output_root="/content/yolo_pill_dataset",
    class_source="auto",
    copy_images=True,
    split_name="train"
)

image_dir = yolo_info["image_dir"]
label_dir = yolo_info["label_dir"]
class_to_id = yolo_info["class_to_id"]

image_files = sorted([
    f for f in os.listdir(image_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

pairs = []
for img_file in image_files:
    stem = os.path.splitext(img_file)[0]
    label_file = f"{stem}.txt"
    if os.path.exists(os.path.join(label_dir, label_file)):
        pairs.append((img_file, label_file))

print("Number of image-label pairs:", len(pairs))

fold_yaml_paths = make_yolo_kfold_dataset(
    pairs=pairs,
    image_dir=image_dir,
    label_dir=label_dir,
    output_root="/content/yolo_pill_dataset_kfold",
    class_to_id=class_to_id,
    n_splits=5,
    random_state=42,
    shuffle=True
)

Number of image files: 232
Sample image files: ['K-001900-016548-019607-029451_0_2_0_2_70_000_200.png', 'K-001900-016548-019607-029451_0_2_0_2_75_000_200.png', 'K-001900-016548-019607-029451_0_2_0_2_90_000_200.png', 'K-001900-016548-019607-033009_0_2_0_2_70_000_200.png', 'K-001900-016548-019607-033009_0_2_0_2_75_000_200.png']
annotations_df shape: (763, 19)
YOLO dataset created at: /content/yolo_pill_dataset
Valid images: 232
Valid boxes: 763
Number of classes: 56
Number of image-label pairs: 232
Created fold yaml files:
 - /content/yolo_pill_dataset_kfold/fold_1/dataset.yaml
 - /content/yolo_pill_dataset_kfold/fold_2/dataset.yaml
 - /content/yolo_pill_dataset_kfold/fold_3/dataset.yaml
 - /content/yolo_pill_dataset_kfold/fold_4/dataset.yaml
 - /content/yolo_pill_dataset_kfold/fold_5/dataset.yaml


## 4. Scratch 실험 설정

In [51]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

SELECTED_FOLD = 1
AUG_MODE = "none"   # "none" / "light" / "mid"
USE_RESIZE = False
BATCH_SIZE = 16
NUM_WORKERS = 2
EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 1e-4

B = 1
CONF_TH = 0.5

FOLD_ROOT = os.path.dirname(fold_yaml_paths[SELECTED_FOLD - 1])
YAML_PATH = os.path.join(FOLD_ROOT, "dataset.yaml")
TRAIN_IMAGE_DIR = os.path.join(FOLD_ROOT, "images", "train")
TRAIN_LABEL_DIR = os.path.join(FOLD_ROOT, "labels", "train")
VAL_IMAGE_DIR   = os.path.join(FOLD_ROOT, "images", "val")
VAL_LABEL_DIR   = os.path.join(FOLD_ROOT, "labels", "val")

with open(YAML_PATH, "r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)

if isinstance(data_yaml["names"], dict):
    CLASS_NAMES = [data_yaml["names"][i] for i in sorted(data_yaml["names"].keys())]
else:
    CLASS_NAMES = data_yaml["names"]

NUM_CLASSES = len(CLASS_NAMES)

sample_image_path = sorted([p for p in Path(TRAIN_IMAGE_DIR).glob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']])[0]
sample_image = cv2.imread(str(sample_image_path))
IMG_H, IMG_W = sample_image.shape[:2]


def downsample_size(size, n_down=5):
    for _ in range(n_down):
        size = (size + 1) // 2  # ceil(size / 2)
    return size

GRID_H = downsample_size(IMG_H, n_down=5)
GRID_W = downsample_size(IMG_W, n_down=5)

print("YAML_PATH       :", YAML_PATH)
print("TRAIN_IMAGE_DIR :", TRAIN_IMAGE_DIR)
print("TRAIN_LABEL_DIR :", TRAIN_LABEL_DIR)
print("VAL_IMAGE_DIR   :", VAL_IMAGE_DIR)
print("VAL_LABEL_DIR   :", VAL_LABEL_DIR)
print("NUM_CLASSES     :", NUM_CLASSES)
print("CLASS_NAMES[:10]:", CLASS_NAMES[:10])
print("USE_RESIZE      :", USE_RESIZE)
print("IMAGE_SIZE      :", (IMG_H, IMG_W))
print("GRID_SIZE       :", (GRID_H, GRID_W))

DEVICE: cuda
YAML_PATH       : /content/yolo_pill_dataset_kfold/fold_1/dataset.yaml
TRAIN_IMAGE_DIR : /content/yolo_pill_dataset_kfold/fold_1/images/train
TRAIN_LABEL_DIR : /content/yolo_pill_dataset_kfold/fold_1/labels/train
VAL_IMAGE_DIR   : /content/yolo_pill_dataset_kfold/fold_1/images/val
VAL_LABEL_DIR   : /content/yolo_pill_dataset_kfold/fold_1/labels/val
NUM_CLASSES     : 56
CLASS_NAMES[:10]: ['가바토파정 100mg', '글리아타민연질캡슐', '글리틴정(콜린알포세레이트)', '기넥신에프정(은행엽엑스)(수출용)', '노바스크정 5mg', '놀텍정 10mg', '뉴로메드정(옥시라세탐)', '다보타민큐정 10mg/병', '동아가바펜틴정 800mg', '라비에트정 20mg']
USE_RESIZE      : False
IMAGE_SIZE      : (1280, 976)
GRID_SIZE       : (40, 31)


## 5. 유틸 / Dataset

In [52]:
def xywhn_to_xyxy(box, img_w, img_h):
    x_c, y_c, w, h = box
    x1 = (x_c - w / 2) * img_w
    y1 = (y_c - h / 2) * img_h
    x2 = (x_c + w / 2) * img_w
    y2 = (y_c + h / 2) * img_h
    return [x1, y1, x2, y2]

def xyxy_to_xywh(box):
    x1, y1, x2, y2 = box
    w = x2 - x1
    h = y2 - y1
    x_c = x1 + w / 2
    y_c = y1 + h / 2
    return [x_c, y_c, w, h]

def clamp_box_xyxy(box, img_w, img_h):
    x1, y1, x2, y2 = box
    x1 = max(0, min(x1, img_w - 1))
    y1 = max(0, min(y1, img_h - 1))
    x2 = max(0, min(x2, img_w - 1))
    y2 = max(0, min(y2, img_h - 1))
    return [x1, y1, x2, y2]

def resize_boxes(boxes, old_w, old_h, new_w, new_h):
    scale_x = new_w / old_w
    scale_y = new_h / old_h
    boxes = boxes.copy()
    boxes[:, [0, 2]] *= scale_x
    boxes[:, [1, 3]] *= scale_y
    return boxes

def pad_to_size(image, target_h, target_w):
    h, w = image.shape[:2]
    pad_bottom = max(0, target_h - h)
    pad_right = max(0, target_w - w)
    if pad_bottom == 0 and pad_right == 0:
        return image
    return cv2.copyMakeBorder(image, 0, pad_bottom, 0, pad_right, borderType=cv2.BORDER_CONSTANT, value=(0, 0, 0))

In [53]:
class SimpleAugment:
    def __init__(self, mode="none"):
        self.mode = mode

    def __call__(self, image, boxes):
        h, w = image.shape[:2]

        if self.mode == "none":
            return image, boxes

        if random.random() < 0.5:
            image = cv2.flip(image, 1)
            if len(boxes) > 0:
                x1 = boxes[:, 0].copy()
                x2 = boxes[:, 2].copy()
                boxes[:, 0] = w - x2
                boxes[:, 2] = w - x1

        if self.mode in ["light", "mid"]:
            hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(np.float32)

            if self.mode == "light":
                sat_scale = random.uniform(0.9, 1.1)
                val_scale = random.uniform(0.9, 1.1)
            else:
                sat_scale = random.uniform(0.85, 1.15)
                val_scale = random.uniform(0.85, 1.15)

            hsv[..., 1] *= sat_scale
            hsv[..., 2] *= val_scale
            hsv[..., 1] = np.clip(hsv[..., 1], 0, 255)
            hsv[..., 2] = np.clip(hsv[..., 2], 0, 255)
            image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

        return image, boxes

class YoloDetectionDataset(Dataset):
    def __init__(self, image_dir, label_dir, image_h, image_w, grid_h, grid_w, C=56, augment_mode="none", use_resize=False):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.image_h = image_h
        self.image_w = image_w
        self.grid_h = grid_h
        self.grid_w = grid_w
        self.C = C
        self.use_resize = use_resize
        self.augment = SimpleAugment(augment_mode)

        self.image_paths = sorted([
            p for p in self.image_dir.glob("*")
            if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]
        ])

    def __len__(self):
        return len(self.image_paths)

    def read_label(self, label_path, img_w, img_h):
        boxes = []
        labels = []

        if not label_path.exists():
            return np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.int64)

        with open(label_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            cls_id = int(parts[0])
            x_c, y_c, w, h = map(float, parts[1:])
            box = xywhn_to_xyxy([x_c, y_c, w, h], img_w, img_h)
            box = clamp_box_xyxy(box, img_w, img_h)

            boxes.append(box)
            labels.append(cls_id)

        if len(boxes) == 0:
            return np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.int64)

        return np.array(boxes, dtype=np.float32), np.array(labels, dtype=np.int64)

    def build_target(self, boxes, labels, img_h, img_w):
        target = np.zeros((self.grid_h, self.grid_w, 5 + self.C), dtype=np.float32)

        for box, cls_id in zip(boxes, labels):
            x_c, y_c, w, h = xyxy_to_xywh(box)

            x_c_norm = x_c / img_w
            y_c_norm = y_c / img_h
            w_norm = w / img_w
            h_norm = h / img_h

            grid_x = min(int(x_c_norm * self.grid_w), self.grid_w - 1)
            grid_y = min(int(y_c_norm * self.grid_h), self.grid_h - 1)

            local_x = x_c_norm * self.grid_w - grid_x
            local_y = y_c_norm * self.grid_h - grid_y

            if target[grid_y, grid_x, 0] == 1:
                continue

            target[grid_y, grid_x, 0] = 1.0
            target[grid_y, grid_x, 1] = local_x
            target[grid_y, grid_x, 2] = local_y
            target[grid_y, grid_x, 3] = w_norm
            target[grid_y, grid_x, 4] = h_norm
            target[grid_y, grid_x, 5 + cls_id] = 1.0

        return target

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label_path = self.label_dir / f"{image_path.stem}.txt"

        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h0, w0 = image.shape[:2]
        boxes, labels = self.read_label(label_path, w0, h0)

        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        image, boxes = self.augment(image, boxes)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.use_resize:
            image_out = cv2.resize(image, (self.image_w, self.image_h))
            if len(boxes) > 0:
                boxes = resize_boxes(boxes, w0, h0, self.image_w, self.image_h)
            current_h, current_w = self.image_h, self.image_w
        else:
            image_out = pad_to_size(image, self.image_h, self.image_w)
            current_h, current_w = h0, w0

        target = self.build_target(boxes, labels, current_h, current_w)

        image_tensor = torch.from_numpy(image_out).permute(2, 0, 1).float() / 255.0
        target_tensor = torch.from_numpy(target)

        sample = {
            "image": image_tensor,
            "target": target_tensor,
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.long),
            "image_path": str(image_path),
            "orig_size": (h0, w0)
        }
        return sample

In [54]:
def collate_fn(batch):
    heights = [b["image"].shape[1] for b in batch]
    widths = [b["image"].shape[2] for b in batch]
    if len(set(heights)) != 1 or len(set(widths)) != 1:
        raise ValueError(f"Images in a batch must have same size, got heights={set(heights)}, widths={set(widths)}")

    images = torch.stack([b["image"] for b in batch], dim=0)
    targets = torch.stack([b["target"] for b in batch], dim=0)

    boxes = [b["boxes"] for b in batch]
    labels = [b["labels"] for b in batch]
    image_paths = [b["image_path"] for b in batch]
    orig_sizes = [b["orig_size"] for b in batch]

    return {
        "images": images,
        "targets": targets,
        "boxes": boxes,
        "labels": labels,
        "image_paths": image_paths,
        "orig_sizes": orig_sizes
    }

train_dataset = YoloDetectionDataset(
    image_dir=TRAIN_IMAGE_DIR,
    label_dir=TRAIN_LABEL_DIR,
    image_h=IMG_H,
    image_w=IMG_W,
    grid_h=GRID_H,
    grid_w=GRID_W,
    C=NUM_CLASSES,
    augment_mode=AUG_MODE,
    use_resize=USE_RESIZE
)

val_dataset = YoloDetectionDataset(
    image_dir=VAL_IMAGE_DIR,
    label_dir=VAL_LABEL_DIR,
    image_h=IMG_H,
    image_w=IMG_W,
    grid_h=GRID_H,
    grid_w=GRID_W,
    C=NUM_CLASSES,
    augment_mode="none",
    use_resize=USE_RESIZE
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

print("Train dataset size:", len(train_dataset))
print("Val dataset size  :", len(val_dataset))

Train dataset size: 185
Val dataset size  : 47


## 6. Scratch Detector

In [55]:
class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

class Detector(nn.Module):
    def __init__(self, num_classes, grid_h, grid_w, B=2):
        super().__init__()
        self.grid_h = grid_h
        self.grid_w = grid_w
        self.B = B
        self.C = num_classes
        self.out_dim = B * 5 + num_classes

        self.backbone = nn.Sequential(
            ConvBNAct(3, 32, 3, 2, 1),
            ConvBNAct(32, 64, 3, 2, 1),
            ConvBNAct(64, 128, 3, 2, 1),
            ConvBNAct(128, 128, 3, 1, 1),
            ConvBNAct(128, 256, 3, 2, 1),
            ConvBNAct(256, 256, 3, 1, 1),
            ConvBNAct(256, 512, 3, 2, 1),
            ConvBNAct(512, 512, 3, 1, 1),
            ConvBNAct(512, 512, 3, 1, 1),
        )

        self.head = nn.Sequential(
            ConvBNAct(512, 256, 3, 1, 1),
            nn.Conv2d(256, self.out_dim, kernel_size=1)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.head(x)
        x = x.permute(0, 2, 3, 1)
        return x

model = Detector(num_classes=NUM_CLASSES, grid_h=GRID_H, grid_w=GRID_W, B=B).to(DEVICE)
dummy = torch.randn(2, 3, IMG_H, IMG_W).to(DEVICE)
out = model(dummy)
print(out.shape)

torch.Size([2, 40, 31, 61])


In [56]:
class FocalBCELoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        loss = alpha_t * (1 - pt).pow(self.gamma) * bce

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss

In [57]:
class DetectionLoss(nn.Module):
    def __init__(self, grid_h=20, grid_w=20, B=2, C=56, lambda_obj=1.0, lambda_box=7.5, lambda_cls=1.0):
        super().__init__()
        self.grid_h = grid_h
        self.grid_w = grid_w
        self.B = B
        self.C = C
        self.lambda_obj = lambda_obj
        self.lambda_box = lambda_box
        self.lambda_cls = lambda_cls

    def forward(self, pred, target):
        N = pred.size(0)

        pred_boxes = pred[..., :self.B * 5].view(N, self.grid_h, self.grid_w, self.B, 5)
        pred_cls = pred[..., self.B * 5:]

        obj_target = target[..., 0]
        tx_ty_wh_target = target[..., 1:5]
        cls_target = target[..., 5:]

        pred_box0 = pred_boxes[..., 0, :]
        pred_obj = pred_box0[..., 0]
        pred_xywh = pred_box0[..., 1:5]

        obj_loss = F.binary_cross_entropy_with_logits(pred_obj, obj_target, reduction="mean")

        obj_mask = obj_target.unsqueeze(-1)
        if obj_mask.sum() > 0:
            box_loss = F.l1_loss(
                torch.sigmoid(pred_xywh) * obj_mask,
                tx_ty_wh_target * obj_mask,
                reduction="sum"
            ) / (obj_mask.sum() + 1e-6)

            cls_indices = cls_target.argmax(dim=-1)
            cls_loss_all = F.cross_entropy(
                pred_cls.reshape(-1, self.C),
                cls_indices.reshape(-1),
                reduction="none"
            ).reshape(N, self.grid_h, self.grid_w)

            cls_loss = (cls_loss_all * obj_target).sum() / (obj_target.sum() + 1e-6)
        else:
            box_loss = torch.tensor(0.0, device=pred.device)
            cls_loss = torch.tensor(0.0, device=pred.device)

        total_loss = (
            self.lambda_obj * obj_loss +
            self.lambda_box * box_loss +
            self.lambda_cls * cls_loss
        )

        return total_loss, {
            "loss_total": total_loss.item(),
            "loss_obj": obj_loss.item(),
            "loss_box": box_loss.item(),
            "loss_cls": cls_loss.item()
        }

criterion = DetectionLoss(grid_h=GRID_H, grid_w=GRID_W, B=B, C=NUM_CLASSES, lambda_obj=1.0, lambda_box=7.5, lambda_cls=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [58]:
class DetectionFocalLoss(nn.Module):
    def __init__(
        self,
        grid_h=20,
        grid_w=20,
        B=2,
        C=56,
        lambda_obj=1.0,
        lambda_box=7.5,
        lambda_cls=1.0,
        focal_alpha=0.25,
        focal_gamma=2.0
    ):
        super().__init__()
        self.grid_h = grid_h
        self.grid_w = grid_w
        self.B = B
        self.C = C
        self.lambda_obj = lambda_obj
        self.lambda_box = lambda_box
        self.lambda_cls = lambda_cls

        self.obj_criterion = FocalBCELoss(alpha=focal_alpha, gamma=focal_gamma, reduction="mean")
        self.box_criterion = nn.SmoothL1Loss(reduction="none")

    def forward(self, pred, target):
        N = pred.size(0)

        pred_boxes = pred[..., :self.B * 5].view(N, self.grid_h, self.grid_w, self.B, 5)
        pred_cls = pred[..., self.B * 5:]

        obj_target = target[..., 0]
        tx_ty_wh_target = target[..., 1:5]
        cls_target = target[..., 5:]

        pred_box0 = pred_boxes[..., 0, :]
        pred_obj = pred_box0[..., 0]
        pred_xywh = pred_box0[..., 1:5]

        obj_loss = self.obj_criterion(pred_obj, obj_target)
        obj_mask = obj_target.unsqueeze(-1)

        if obj_mask.sum() > 0:
            pred_xywh_sigmoid = torch.sigmoid(pred_xywh)
            box_loss_all = self.box_criterion(pred_xywh_sigmoid, tx_ty_wh_target)
            box_loss = (box_loss_all * obj_mask).sum() / (obj_mask.sum() + 1e-6)

            cls_indices = cls_target.argmax(dim=-1)
            cls_loss_all = F.cross_entropy(
                pred_cls.reshape(-1, self.C),
                cls_indices.reshape(-1),
                reduction="none"
            ).reshape(N, self.grid_h, self.grid_w)

            cls_loss = (cls_loss_all * obj_target).sum() / (obj_target.sum() + 1e-6)
        else:
            box_loss = torch.tensor(0.0, device=pred.device)
            cls_loss = torch.tensor(0.0, device=pred.device)

        total_loss = (
            self.lambda_obj * obj_loss +
            self.lambda_box * box_loss +
            self.lambda_cls * cls_loss
        )

        return total_loss, {
            "loss_total": total_loss.item(),
            "loss_obj": obj_loss.item(),
            "loss_box": box_loss.item(),
            "loss_cls": cls_loss.item()
        }

In [59]:
criterion = DetectionFocalLoss(
    grid_h=GRID_H,
    grid_w=GRID_W,
    B=B,
    C=NUM_CLASSES,
    lambda_obj=1.0,
    lambda_box=8.0,
    lambda_cls=1.0,
    focal_alpha=0.25,
    focal_gamma=2.0
)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 7. Decode / Train / Eval

In [60]:
def decode_predictions(pred, conf_thresh=0.3, img_h=IMG_H, img_w=IMG_W):
    pred = pred.detach().cpu()
    pred_boxes = pred[..., :B * 5].view(GRID_H, GRID_W, B, 5)
    pred_cls = pred[..., B * 5:]

    boxes = []
    scores = []
    labels = []

    class_probs = torch.softmax(pred_cls, dim=-1)

    for gy in range(GRID_H):
        for gx in range(GRID_W):
            for b in range(B):
                obj_logit = pred_boxes[gy, gx, b, 0]
                xywh = torch.sigmoid(pred_boxes[gy, gx, b, 1:5])

                obj_score = torch.sigmoid(obj_logit).item()
                cls_score, cls_id = class_probs[gy, gx].max(dim=-1)

                score = obj_score * cls_score.item()
                if score < conf_thresh:
                    continue

                local_x, local_y, w_norm, h_norm = xywh.tolist()

                x_c = (gx + local_x) / GRID_W * img_w
                y_c = (gy + local_y) / GRID_H * img_h
                w = w_norm * img_w
                h = h_norm * img_h

                x1 = max(0.0, x_c - w / 2)
                y1 = max(0.0, y_c - h / 2)
                x2 = min(float(img_w - 1), x_c + w / 2)
                y2 = min(float(img_h - 1), y_c + h / 2)

                boxes.append([x1, y1, x2, y2])
                scores.append(score)
                labels.append(cls_id.item())

    if len(boxes) == 0:
        return torch.zeros((0, 4)), torch.zeros((0,)), torch.zeros((0,), dtype=torch.long)

    boxes = torch.tensor(boxes, dtype=torch.float32)
    scores = torch.tensor(scores, dtype=torch.float32)
    labels = torch.tensor(labels, dtype=torch.long)

    keep = nms(boxes, scores, iou_threshold=0.5)
    return boxes[keep], scores[keep], labels[keep]

In [61]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running = {"loss_total": 0.0, "loss_obj": 0.0, "loss_box": 0.0, "loss_cls": 0.0}
    count = 0

    for batch in loader:
        images = batch["images"].to(device)
        targets = batch["targets"].to(device)

        optimizer.zero_grad()
        preds = model(images)
        loss, loss_dict = criterion(preds, targets)
        loss.backward()
        optimizer.step()

        bs = images.size(0)
        count += bs
        for k in running:
            running[k] += loss_dict[k] * bs

    for k in running:
        running[k] /= count
    return running

@torch.no_grad()
def evaluate(model, loader, criterion, device, conf_thresh=0.3):
    model.eval()
    running = {"loss_total": 0.0, "loss_obj": 0.0, "loss_box": 0.0, "loss_cls": 0.0}
    count = 0

    metric_map = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", iou_thresholds=[0.75])

    for batch in loader:
        images = batch["images"].to(device)
        targets = batch["targets"].to(device)

        preds = model(images)
        loss, loss_dict = criterion(preds, targets)

        bs = images.size(0)
        count += bs
        for k in running:
            running[k] += loss_dict[k] * bs

        for i in range(bs):
            img_h, img_w = batch["orig_sizes"][i]
            pred_boxes, pred_scores, pred_labels = decode_predictions(preds[i], conf_thresh=conf_thresh, img_h=img_h, img_w=img_w)

            gt_boxes = batch["boxes"][i]
            gt_labels = batch["labels"][i]

            preds_for_metric = {
                "boxes": pred_boxes,
                "scores": pred_scores,
                "labels": pred_labels
            }
            target_for_metric = {
                "boxes": gt_boxes,
                "labels": gt_labels
            }
            metric_map.update([preds_for_metric], [target_for_metric])

    for k in running:
        running[k] /= count

    metric_result = metric_map.compute()
    running["map75"] = metric_result["map"].item()
    return running

## 8. 학습

In [62]:
best_map75 = -1
best_path = f"best_scratch_detector_fold{SELECTED_FOLD}.pth"

train_history = []
val_history = []

for epoch in range(1, EPOCHS + 1):
    train_log = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_log = evaluate(model, val_loader, criterion, DEVICE, conf_thresh=CONF_TH)
    scheduler.step()

    train_history.append(train_log)
    val_history.append(val_log)

    print(
        f"[Epoch {epoch:03d}] "
        f"train_loss={train_log['loss_total']:.4f} | "
        f"val_loss={val_log['loss_total']:.4f} | "
        f"val_map75={val_log['map75']:.4f}"
    )

    if val_log["map75"] > best_map75:
        best_map75 = val_log["map75"]
        torch.save(model.state_dict(), best_path)
        print(f"  -> Best model saved: {best_path}")

[Epoch 001] train_loss=5.1939 | val_loss=50.4166 | val_map75=0.0000
  -> Best model saved: best_scratch_detector_fold1.pth
[Epoch 002] train_loss=4.1339 | val_loss=4.0448 | val_map75=0.0000
[Epoch 003] train_loss=3.7448 | val_loss=4.4195 | val_map75=0.0000
[Epoch 004] train_loss=3.3615 | val_loss=3.9825 | val_map75=0.0000
[Epoch 005] train_loss=3.0539 | val_loss=2.9752 | val_map75=0.0000
[Epoch 006] train_loss=2.7972 | val_loss=2.9045 | val_map75=0.0000
[Epoch 007] train_loss=2.5622 | val_loss=2.5968 | val_map75=0.0000


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79f01f3dc360>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79f01f3dc360>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

[Epoch 008] train_loss=2.3215 | val_loss=2.5551 | val_map75=0.0000
[Epoch 009] train_loss=2.3145 | val_loss=2.3250 | val_map75=0.0000
[Epoch 010] train_loss=2.2284 | val_loss=2.4428 | val_map75=0.0000
[Epoch 011] train_loss=1.9156 | val_loss=2.0896 | val_map75=0.0000
[Epoch 012] train_loss=1.8076 | val_loss=2.3628 | val_map75=0.0000
[Epoch 013] train_loss=1.6712 | val_loss=1.9740 | val_map75=0.0000
[Epoch 014] train_loss=1.5624 | val_loss=1.6783 | val_map75=0.0000
[Epoch 015] train_loss=1.4659 | val_loss=2.1223 | val_map75=0.0000
[Epoch 016] train_loss=1.5112 | val_loss=2.4783 | val_map75=0.0000
[Epoch 017] train_loss=1.3035 | val_loss=3.7981 | val_map75=0.0000
[Epoch 018] train_loss=1.1982 | val_loss=1.8414 | val_map75=0.0000
[Epoch 019] train_loss=1.4574 | val_loss=1.5613 | val_map75=0.0057
  -> Best model saved: best_scratch_detector_fold1.pth


KeyboardInterrupt: 

In [ ]:
train_losses = [x["loss_total"] for x in train_history]
val_losses = [x["loss_total"] for x in val_history]
val_map75s = [x["map75"] for x in val_history]

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="train_loss")
plt.plot(val_losses, label="val_loss")
plt.legend()
plt.title("Loss Curve")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(val_map75s, label="val_map75")
plt.legend()
plt.title("Validation mAP@0.75")
plt.show()

## 9. 시각화 / 최종 평가

In [ ]:
def draw_boxes(image, boxes, labels, scores=None, class_names=None):
    image = image.copy()

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        cls_id = int(labels[i])

        text = f"{class_names[cls_id]}"
        if scores is not None:
            text += f" {scores[i]:.2f}"

        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            image, text, (x1, max(15, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1
        )

    return image


model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.to(DEVICE)
model.eval()

In [ ]:
@torch.no_grad()
def visualize_predictions(dataset, model, device, num_samples=5, conf_thresh=0.3):
    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))

    for idx in indices:
        sample = dataset[idx]
        image = sample["image"].unsqueeze(0).to(device)
        gt_boxes = sample["boxes"].numpy()
        gt_labels = sample["labels"].numpy()
        img_h, img_w = sample["orig_size"]

        pred = model(image)[0]
        pred_boxes, pred_scores, pred_labels = decode_predictions(pred, conf_thresh=conf_thresh, img_h=img_h, img_w=img_w)

        img_np = (sample["image"].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        img_np = img_np[:img_h, :img_w]

        gt_img = draw_boxes(img_np, gt_boxes, gt_labels, class_names=CLASS_NAMES)
        pred_img = draw_boxes(
            img_np,
            pred_boxes.numpy(),
            pred_labels.numpy(),
            pred_scores.numpy(),
            class_names=CLASS_NAMES
        )

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].imshow(gt_img)
        axes[0].set_title("Ground Truth")
        axes[0].axis("off")

        axes[1].imshow(pred_img)
        axes[1].set_title("Prediction")
        axes[1].axis("off")

        plt.show()

visualize_predictions(val_dataset, model, DEVICE, num_samples=5, conf_thresh=CONF_TH)

In [ ]:
@torch.no_grad()
def evaluate_map75_map7595(model, loader, device, conf_thresh=0.3):
    model.eval()

    metric_75 = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", iou_thresholds=[0.75])
    metric_7595 = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        iou_thresholds=[0.75, 0.80, 0.85, 0.90, 0.95]
    )

    for batch in loader:
        images = batch["images"].to(device)
        preds = model(images)

        bs = images.size(0)
        for i in range(bs):
            img_h, img_w = batch["orig_sizes"][i]
            pred_boxes, pred_scores, pred_labels = decode_predictions(preds[i], conf_thresh=conf_thresh, img_h=img_h, img_w=img_w)

            gt_boxes = batch["boxes"][i]
            gt_labels = batch["labels"][i]

            pred_dict = {
                "boxes": pred_boxes,
                "scores": pred_scores,
                "labels": pred_labels
            }
            target_dict = {
                "boxes": gt_boxes,
                "labels": gt_labels
            }

            metric_75.update([pred_dict], [target_dict])
            metric_7595.update([pred_dict], [target_dict])

    res75 = metric_75.compute()
    res7595 = metric_7595.compute()

    print("mAP75    :", res75["map"].item())
    print("mAP75:95 :", res7595["map"].item())

model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.to(DEVICE)
model.eval()
for th in [0.3, 0.4, 0.5]:
    print(f"\n=== conf_thresh={th} ===")
    evaluate_map75_map7595(model, val_loader, DEVICE, conf_thresh=th)